[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/057.%20The%20Periodic%20Review%20%28Base-Stock%29%20Policy%20Problem/P57-Tier-2_executed.ipynb)

---



# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 2: Constructive Base-Stock Optimization by Grid Search

### Purpose and scope

This notebook searches for a low-cost periodic-review policy by evaluating a grid of review periods. For each review period, it calculates a base-stock level from a target service level, evaluates expected weekly costs, and selects the least-cost grid point. A Monte Carlo simulation then provides a finite-horizon operational diagnostic for that policy.

The example concerns syringe inventory at MedSupply Distribution. Demand is modeled as normally distributed with a known weekly mean and standard deviation; lead time and cost parameters are constant. Review periods are allowed to be fractional weeks. The grid search is a heuristic: its result is the least-cost point on the specified grid, not a proof of a continuous global optimum.

### Inputs and outputs

Inputs are the demand parameters, lead time, target service level, ordering cost, holding cost, shortage cost, and the review-period grid. The main outputs are the selected review period, its associated base-stock level, the expected weekly cost breakdown, sensitivity results, and simulation performance measures.

### Modeling assumptions

- Weekly demand has a normal distribution with mean $\\mu$ and standard deviation $\\sigma$. Demand over an interval of length $t$ weeks is modeled with mean $\\mu t$ and standard deviation $\\sigma\\sqrt{t}$.
- Lead time $L$ is known and constant. The protection interval for periodic review is $R+L$.
- The base-stock level is set to the demand quantile associated with the specified service-level target $\\alpha$. It is therefore the level implied by that target, rather than an independent minimizer of the cost function.
- The expected-cost calculation uses average cycle stock, safety stock, and the normal loss function. It is a steady-state approximation, not a detailed representation of every operational event.
- The simulation represents demand as uniform within each simulated week and uses 0.01-week time steps. It permits backorders, charges shortage cost when demand cannot be met immediately, and includes a finite-horizon effect. These details make the simulation an operational diagnostic, not an exact replication of the expected-cost approximation.
- The simulation's period service measure is the fraction of time steps with no unmet demand. It is not the same as the model's target quantile service level, nor is it the unit fill rate.

### Method and mathematical model

A review period $R$ determines both the ordering frequency and the length of the period protected by stock. The grid search evaluates candidate values of $R$ in increasing order. At each point it first computes the target-service base-stock level $S(R)$, then evaluates the expected weekly cost.

Let $z_\\alpha=\\Phi^{-1}(\\alpha)$, where $\\Phi$ and $\\phi$ are the standard normal cumulative distribution and density functions. For a candidate review period $R$, the protection interval is $T=R+L$, and demand over that interval has mean $\\mu T$ and standard deviation $\\sigma\\sqrt{T}$. The base-stock rule is

$$S(R)=\\mu(R+L)+z_\\alpha\\sigma\\sqrt{R+L}.$$

This is a quantile rule: under the normal-demand assumption, it places the specified fraction $\\alpha$ of protection-interval demand at or below the base-stock level.

The expected weekly cost used in the search is

$$C(R,S)=\\frac{K}{R}+h\\left(\\frac{\\mu R}{2}+z_\\alpha\\sigma\\sqrt{R+L}\\right)+\\frac{p}{R}\\sigma\\sqrt{R+L}\\left[\\phi(z)-z\\left(1-\\Phi(z)\\right)\\right],$$

where

$$z=\\frac{S-\\mu(R+L)}{\\sigma\\sqrt{R+L}}.$$

Here, $K$ is the fixed cost per order, $h$ is the holding cost per unit per week, and $p$ is the shortage cost per unit. The first term is ordering cost per week. The holding-cost term uses average cycle stock $\\mu R/2$ plus safety stock. The final term uses the normal expected-shortage (loss) function over the protection interval, converted to a weekly rate by dividing by $R$. In the search, $S=S(R)$, so $z=z_\\alpha$ apart from floating-point rounding.

The grid-search decision is

$$R_{\\mathrm{grid}}\\in\\underset{R\\in\\mathcal{G}}{\\arg\\min}\\ C\\bigl(R,S(R)\\bigr),$$

where $\\mathcal{G}$ is the finite set of review periods tested. If more than one grid point has the same minimum cost, each is a grid optimum; the implementation retains the first minimum encountered. A finer grid can change the reported grid optimum, and increases computation in proportion to the number of candidate review periods.

In the code, `calculate_base_stock_level` implements the quantile rule, `calculate_total_cost` implements the cost equation, and `optimize_policy` evaluates the grid and retains the minimum. The simulation uses the selected values but is not used to alter the optimization result.

### Step-by-step approach

1. **Define the grid:** choose the minimum and maximum review periods and the grid increment.
2. **Calculate the base-stock level:** apply the target-service quantile rule to each candidate period.
3. **Evaluate expected cost:** calculate ordering, holding, and shortage costs per week.
4. **Select the grid minimum:** retain the candidate with the smallest expected weekly cost.
5. **Inspect sensitivity and simulate:** examine nearby grid points, then use a reproducible Monte Carlo simulation as an operational diagnostic.

### Example data

| Parameter | Value | Meaning |
|---|---:|---|
| $\\mu$ | 12,000 units/week | Mean weekly demand |
| $\\sigma$ | 2,400 units/week | Standard deviation of weekly demand |
| $L$ | 0.5 weeks | Lead time |
| $\\alpha$ | 0.95 | Target quantile service level |
| $h$ | $0.15 per unit per week | Holding cost rate |
| $K$ | $75 per order | Fixed ordering cost |
| $p$ | $8 per unit | Shortage cost |

The numerical solution is calculated by the notebook from these inputs and the stated grid; it is not assumed in advance. Results should be interpreted as a grid-based heuristic under the expected-cost model, with the simulation providing a separate finite-horizon diagnostic.

In [ ]:
import warnings
from math import floor, sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
class BaseStockHeuristicOptimizer:
    """Grid-search optimizer for a periodic-review base-stock policy.

    For each candidate review period, the class calculates the base-stock
    level implied by the specified service-level quantile and evaluates
    the expected weekly cost components.
    """

    def __init__(
        self,
        demand_mean,
        demand_std,
        lead_time,
        holding_cost,
        ordering_cost,
        stockout_cost,
        service_level,
    ):
        """Store and validate the parameters used by the model."""
        if demand_mean <= 0 or demand_std <= 0:
            raise ValueError("Demand mean and standard deviation must be positive.")
        if lead_time < 0:
            raise ValueError("Lead time cannot be negative.")
        if holding_cost < 0 or ordering_cost < 0 or stockout_cost < 0:
            raise ValueError("Cost parameters cannot be negative.")
        if not 0 < service_level < 1:
            raise ValueError("Service level must be strictly between 0 and 1.")

        self.mu = demand_mean
        self.sigma = demand_std
        self.L = lead_time
        self.h = holding_cost
        self.K = ordering_cost
        self.p = stockout_cost
        self.alpha = service_level
        self.z_alpha = norm.ppf(service_level)

        print("Initialized heuristic optimizer:")
        print(f"  Weekly demand: N({self.mu:,.0f}, {self.sigma:,.0f}²) units")
        print(f"  Lead time: {self.L:g} weeks")
        print(f"  Target quantile service level: {self.alpha:.3f}")
        print(
            f"  Costs: h=${self.h:g}/unit/week, "
            f"K=${self.K:g}/order, p=${self.p:g}/unit"
        )

    def calculate_base_stock_level(self, review_period):
        """Return the service-level base stock and its demand quantities."""
        if review_period <= 0:
            raise ValueError("Review period must be positive.")

        protection_interval = review_period + self.L
        expected_demand = self.mu * protection_interval
        demand_std = self.sigma * sqrt(protection_interval)
        base_stock = expected_demand + self.z_alpha * demand_std

        return base_stock, protection_interval, expected_demand, demand_std

    def calculate_total_cost(self, review_period, base_stock_level):
        """Evaluate expected weekly ordering, holding, and shortage costs."""
        if review_period <= 0:
            raise ValueError("Review period must be positive.")

        protection_interval = review_period + self.L
        demand_std = self.sigma * sqrt(protection_interval)
        expected_protection_demand = self.mu * protection_interval

        ordering_cost = self.K / review_period

        cycle_stock = self.mu * review_period / 2
        safety_stock = self.z_alpha * demand_std
        expected_inventory = cycle_stock + safety_stock
        holding_cost = self.h * expected_inventory

        normalized_shortage = (
            base_stock_level - expected_protection_demand
        ) / demand_std
        expected_shortage = demand_std * (
            norm.pdf(normalized_shortage)
            - normalized_shortage * norm.sf(normalized_shortage)
        )
        stockout_cost = self.p * expected_shortage / review_period

        total_cost = ordering_cost + holding_cost + stockout_cost
        return total_cost, ordering_cost, holding_cost, stockout_cost

    def optimize_policy(self, min_review=0.1, max_review=4.0, step=0.01):
        """Search the specified review-period grid for its minimum cost."""
        if min_review <= 0 or max_review < min_review or step <= 0:
            raise ValueError(
                "Require 0 < min_review <= max_review and step > 0."
            )

        number_of_steps = floor(
            (max_review - min_review) / step + 1e-12
        )
        review_grid = min_review + step * np.arange(number_of_steps + 1)
        results = []

        print(
            f"\nStarting grid search: R from {min_review:g} to "
            f"{max_review:g} weeks, with step {step:g}."
        )

        for review_period in review_grid:
            base_stock = self.calculate_base_stock_level(review_period)[0]
            total_cost, ord_cost, hold_cost, stock_cost = (
                self.calculate_total_cost(review_period, base_stock)
            )
            results.append(
                {
                    "review_period": review_period,
                    "base_stock_level": base_stock,
                    "total_cost": total_cost,
                    "ordering_cost": ord_cost,
                    "holding_cost": hold_cost,
                    "stockout_cost": stock_cost,
                }
            )

        best_result = min(results, key=lambda result: result["total_cost"])
        return {
            "optimal_review_period": best_result["review_period"],
            "optimal_base_stock_level": best_result["base_stock_level"],
            "minimum_total_cost": best_result["total_cost"],
            "all_results": results,
        }

    def simulate_policy(
        self,
        review_period,
        base_stock_level,
        num_periods=52,
        seed=42,
        ticks_per_week=100,
    ):
        """Simulate the policy in discrete time as a finite-horizon diagnostic.

        Weekly normal demand is spread uniformly across the simulation ticks
        in that week. Reviews and deliveries are scheduled on the same time
        grid; unmet demand becomes backlog and is recorded when it occurs.
        """
        if review_period <= 0 or base_stock_level < 0:
            raise ValueError("Review period must be positive and base stock nonnegative.")
        if num_periods <= 0 or ticks_per_week <= 0:
            raise ValueError("Simulation length and ticks per week must be positive.")

        tick_length = 1 / ticks_per_week
        review_steps = max(1, int(round(review_period * ticks_per_week)))
        lead_steps = int(round(self.L * ticks_per_week))
        simulated_review_period = review_steps * tick_length
        simulated_lead_time = lead_steps * tick_length

        rng = np.random.RandomState(seed)
        weekly_demand = np.maximum(
            0,
            rng.normal(self.mu, self.sigma, size=num_periods),
        )
        demand_by_tick = np.repeat(
            weekly_demand / ticks_per_week,
            ticks_per_week,
        )
        total_steps = len(demand_by_tick)

        # The arrival array makes each order's lead-time delay explicit.
        arrivals = np.zeros(total_steps + lead_steps + 1)
        net_inventory = base_stock_level
        pipeline_inventory = 0.0
        inventory_levels = []
        order_quantities = []
        demands = []
        shortage_quantities = []
        stockout_indicators = []
        order_count = 0

        for tick, demand in enumerate(demand_by_tick):
            receipt = arrivals[tick]
            net_inventory += receipt
            pipeline_inventory -= receipt

            shortage = max(0.0, demand - max(net_inventory, 0.0))
            net_inventory -= demand

            demands.append(demand)
            shortage_quantities.append(shortage)
            stockout_indicators.append(int(shortage > 0))
            inventory_levels.append(max(net_inventory, 0.0))

            if tick % review_steps == 0:
                inventory_position = net_inventory + pipeline_inventory
                order_quantity = max(0.0, base_stock_level - inventory_position)
                order_count += int(order_quantity > 0)
                order_quantities.append(order_quantity)

                if order_quantity > 0 and lead_steps > 0:
                    arrivals[tick + lead_steps] += order_quantity
                    pipeline_inventory += order_quantity
                elif order_quantity > 0:
                    net_inventory += order_quantity
            else:
                order_quantities.append(0.0)

        inventory_levels = np.asarray(inventory_levels)
        demands = np.asarray(demands)
        shortage_quantities = np.asarray(shortage_quantities)
        stockout_periods = int(sum(stockout_indicators))
        total_demand = float(demands.sum())
        total_shortage = float(shortage_quantities.sum())
        total_order_quantity = float(sum(order_quantities))
        average_order_size = (
            total_order_quantity / order_count if order_count else 0.0
        )

        return {
            "inventory_levels": inventory_levels,
            "order_quantities": order_quantities,
            "demands": demands,
            "shortage_quantities": shortage_quantities,
            "avg_inventory": float(inventory_levels.mean()),
            "stockout_periods": stockout_periods,
            "service_level": 1 - stockout_periods / total_steps,
            "fill_rate": (
                1 - total_shortage / total_demand if total_demand else 1.0
            ),
            "total_orders": total_order_quantity,
            "order_count": order_count,
            "avg_order_size": average_order_size,
            "total_shortage": total_shortage,
            "total_demand": total_demand,
            "simulated_review_period": simulated_review_period,
            "simulated_lead_time": simulated_lead_time,
            "total_steps": total_steps,
            "num_periods": num_periods,
        }


print("BaseStockHeuristicOptimizer class defined successfully.")

In [ ]:
# Initialize the optimizer with the MedSupply example data.
heuristic_optimizer = BaseStockHeuristicOptimizer(
    demand_mean=12000,
    demand_std=2400,
    lead_time=0.5,
    holding_cost=0.15,
    ordering_cost=75,
    stockout_cost=8,
    service_level=0.95,
)

In [ ]:
# Search review periods from 0.50 to 3.00 weeks in 0.01-week increments.
heuristic_result = heuristic_optimizer.optimize_policy(
    min_review=0.5,
    max_review=3.0,
    step=0.01,
)

optimal_R = heuristic_result["optimal_review_period"]
optimal_S = heuristic_result["optimal_base_stock_level"]
minimum_cost = heuristic_result["minimum_total_cost"]

total_cost, ord_cost, hold_cost, stock_cost = (
    heuristic_optimizer.calculate_total_cost(optimal_R, optimal_S)
)

assert np.isclose(total_cost, minimum_cost, rtol=1e-10, atol=1e-8)
assert np.isclose(
    ord_cost + hold_cost + stock_cost,
    total_cost,
    rtol=1e-10,
    atol=1e-8,
)

print("\n" + "=" * 60)
print("GRID-SEARCH OPTIMIZATION RESULTS")
print("=" * 60)
print(f"Least-cost grid review period: {optimal_R:.2f} weeks")
print(f"Associated base-stock level: {optimal_S:,.0f} units")
print(f"Expected total cost: ${minimum_cost:,.2f} per week")
print("\nEXPECTED WEEKLY COST BREAKDOWN")
print(f"Ordering: ${ord_cost:,.2f} ({ord_cost / total_cost:.1%})")
print(f"Holding:  ${hold_cost:,.2f} ({hold_cost / total_cost:.1%})")
print(f"Shortage: ${stock_cost:,.2f} ({stock_cost / total_cost:.1%})")

# Evaluate the same cost model at a one-week review period for comparison.
tier1_R = 1.0
tier1_S = heuristic_optimizer.calculate_base_stock_level(tier1_R)[0]
tier1_cost = heuristic_optimizer.calculate_total_cost(tier1_R, tier1_S)[0]
cost_savings = tier1_cost - minimum_cost
cost_savings_pct = cost_savings / tier1_cost * 100

print("\nCOMPARISON WITH A ONE-WEEK REVIEW PERIOD")
print(f"One-week review cost: ${tier1_cost:,.2f} per week")
print(f"Least-cost grid result: ${minimum_cost:,.2f} per week")
print(
    f"Difference: ${cost_savings:,.2f} per week "
    f"({cost_savings_pct:.1f}% of the one-week review cost)"
)

In [ ]:
# Tabulate the best grid points and examine costs near the selected period.
results_df = pd.DataFrame(heuristic_result["all_results"])
top_solutions = results_df.nsmallest(5, "total_cost").copy()
top_solutions["cost_increase_pct"] = (
    (top_solutions["total_cost"] - minimum_cost) / minimum_cost * 100
)

print("FIVE LOWEST-COST GRID POINTS")
print("=" * 80)
display_columns = [
    "review_period",
    "base_stock_level",
    "total_cost",
    "ordering_cost",
    "holding_cost",
    "stockout_cost",
]
print(top_solutions[display_columns].round(2).to_string(index=False))

sensitivity_range = 0.3
sensitivity_results = results_df[
    results_df["review_period"].between(
        optimal_R - sensitivity_range,
        optimal_R + sensitivity_range,
    )
].copy()
sensitivity_results["cost_increase_pct"] = (
    (sensitivity_results["total_cost"] - minimum_cost) / minimum_cost * 100
)

print("\nCOST SENSITIVITY NEAR THE SELECTED GRID POINT")
print(
    f"Examined review-period range: "
    f"[{sensitivity_results['review_period'].min():.2f}, "
    f"{sensitivity_results['review_period'].max():.2f}] weeks"
)
print(
    f"Cost range: ${sensitivity_results['total_cost'].min():,.2f} "
    f"to ${sensitivity_results['total_cost'].max():,.2f} per week"
)
print(
    f"Largest increase in this range: "
    f"{sensitivity_results['cost_increase_pct'].max():.1f}%"
)

flat_region = sensitivity_results[
    sensitivity_results["cost_increase_pct"] <= 1.0
]
if not flat_region.empty:
    print(
        "Review periods within 1% of the grid minimum in this range: "
        f"{flat_region['review_period'].min():.2f} to "
        f"{flat_region['review_period'].max():.2f} weeks"
    )
else:
    print("No review period in this range is within 1% of the grid minimum.")

In [ ]:
# Simulate the selected policy using the fixed seed for reproducibility.
simulation_result = heuristic_optimizer.simulate_policy(
    review_period=optimal_R,
    base_stock_level=optimal_S,
    num_periods=52,
    seed=42,
)

simulated_weeks = simulation_result["num_periods"]
sim_holding_cost = (
    simulation_result["avg_inventory"] * heuristic_optimizer.h
)
sim_ordering_cost = (
    simulation_result["order_count"]
    * heuristic_optimizer.K
    / simulated_weeks
)
sim_stockout_cost = (
    simulation_result["total_shortage"]
    * heuristic_optimizer.p
    / simulated_weeks
)
sim_total_cost = (
    sim_holding_cost + sim_ordering_cost + sim_stockout_cost
)

print("FINITE-HORIZON SIMULATION DIAGNOSTIC")
print("=" * 52)
print(
    f"Simulated horizon: {simulated_weeks} weeks "
    f"({simulation_result['total_steps']:,} time steps)"
)
print(
    f"Review period represented on the simulation grid: "
    f"{simulation_result['simulated_review_period']:.2f} weeks"
)
print(
    f"Lead time represented on the simulation grid: "
    f"{simulation_result['simulated_lead_time']:.2f} weeks"
)
print(f"Average on-hand inventory: {simulation_result['avg_inventory']:,.0f} units")
print(
    f"No-shortage time-step fraction: "
    f"{simulation_result['service_level']:.4f} "
    f"({simulation_result['service_level']:.2%})"
)
print(
    f"Unit fill rate: {simulation_result['fill_rate']:.4f} "
    f"({simulation_result['fill_rate']:.2%})"
)
print(
    f"Time steps with unmet demand: {simulation_result['stockout_periods']} "
    f"of {simulation_result['total_steps']:,}"
)
print(f"Orders placed: {simulation_result['order_count']}")
print(f"Total units ordered: {simulation_result['total_orders']:,.0f}")
print(f"Average order size: {simulation_result['avg_order_size']:,.0f} units")

print("\nSIMULATED COST RATES")
print(f"Holding:  ${sim_holding_cost:,.2f} per week")
print(f"Ordering: ${sim_ordering_cost:,.2f} per week")
print(f"Shortage: ${sim_stockout_cost:,.2f} per week")
print(f"Total:    ${sim_total_cost:,.2f} per week")

print("\nEXPECTED-COST MODEL COMPARISON")
print(f"Expected-cost model: ${minimum_cost:,.2f} per week")
print(f"Simulation estimate: ${sim_total_cost:,.2f} per week")
print(
    "Interpret the difference in light of the finite simulation horizon, "
    "discrete time grid, and the distinct cost and service measures."
)

In [ ]:
# Plot the grid-search cost curve, cost components, base-stock rule, and comparison.
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    2,
    2,
    figsize=(16, 12),
)
fig.suptitle(
    "Periodic-Review Base-Stock Policy: Grid-Search Results",
    fontsize=16,
    fontweight="bold",
)

ax1.plot(
    results_df["review_period"],
    results_df["total_cost"],
    color="blue",
    linewidth=2,
    alpha=0.75,
    label="Expected weekly cost",
)
ax1.axvline(
    x=optimal_R,
    color="red",
    linestyle="--",
    label=f"Selected grid point: R = {optimal_R:.2f} weeks",
)
ax1.set_xlabel("Review period, R (weeks)")
ax1.set_ylabel("Expected cost ($ per week)")
ax1.set_title("Expected Cost versus Review Period")
ax1.legend()
ax1.grid(True, alpha=0.3)

optimal_row = results_df.loc[
    np.isclose(results_df["review_period"], optimal_R)
].iloc[0]
costs = [
    optimal_row["ordering_cost"],
    optimal_row["holding_cost"],
    optimal_row["stockout_cost"],
]
labels = ["Ordering", "Holding", "Shortage"]
colors = ["#ff9999", "#66b3ff", "#99ff99"]
ax2.pie(
    costs,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
)
ax2.set_title(f"Expected Cost Components\n(R = {optimal_R:.2f} weeks)")

ax3.plot(
    results_df["review_period"],
    results_df["base_stock_level"],
    color="green",
    linewidth=2,
    label="Service-level base stock",
)
ax3.axvline(
    x=optimal_R,
    color="red",
    linestyle="--",
    label=f"Selected R = {optimal_R:.2f} weeks",
)
ax3.axhline(
    y=optimal_S,
    color="darkorange",
    linestyle=":",
    label=f"Associated S = {optimal_S:,.0f} units",
)
ax3.set_xlabel("Review period, R (weeks)")
ax3.set_ylabel("Base-stock level, S (units)")
ax3.set_title("Base-Stock Level versus Review Period")
ax3.legend()
ax3.grid(True, alpha=0.3)

comparison_df = pd.DataFrame(
    {
        "Policy": ["One-week review", "Least-cost grid point"],
        "Expected weekly cost ($)": [tier1_cost, minimum_cost],
        "Review period (weeks)": [tier1_R, optimal_R],
        "Base stock (units)": [tier1_S, optimal_S],
    }
)
x = np.arange(len(comparison_df))
width = 0.25
ax4.bar(
    x - width,
    comparison_df["Expected weekly cost ($)"],
    width,
    label="Cost ($/week)",
)
ax4.bar(
    x,
    comparison_df["Review period (weeks)"] * 1000,
    width,
    label="Review period (weeks × 1,000)",
)
ax4.bar(
    x + width,
    comparison_df["Base stock (units)"] / 50,
    width,
    label="Base stock (units ÷ 50)",
)
ax4.set_xlabel("Policy")
ax4.set_ylabel("Displayed value (scaling is stated in the legend)")
ax4.set_title("One-Week Review and Selected Grid Point")
ax4.set_xticks(x)
ax4.set_xticklabels(comparison_df["Policy"])
ax4.legend()
ax4.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("The plots display the grid-search trade-off and the associated policy quantities.")

In [ ]:
# Summarize demand, inventory, ordering, and service measures from the simulation.
inventory_ts = simulation_result["inventory_levels"]
demand_ts = simulation_result["demands"]
order_ts = simulation_result["order_quantities"]
shortage_ts = simulation_result["shortage_quantities"]

inventory_stats = {
    "mean": np.mean(inventory_ts),
    "standard deviation": np.std(inventory_ts),
    "minimum": np.min(inventory_ts),
    "maximum": np.max(inventory_ts),
    "median": np.median(inventory_ts),
}
demand_stats = {
    "mean": np.mean(demand_ts),
    "standard deviation": np.std(demand_ts),
    "minimum": np.min(demand_ts),
    "maximum": np.max(demand_ts),
    "median": np.median(demand_ts),
}

print("DETAILED SIMULATION ANALYSIS")
print("=" * 52)
print("ON-HAND INVENTORY STATISTICS (units):")
for statistic, value in inventory_stats.items():
    print(f"  {statistic.capitalize()}: {value:,.2f}")

print("\nDEMAND PER SIMULATION TIME STEP (units):")
for statistic, value in demand_stats.items():
    print(f"  {statistic.capitalize()}: {value:,.2f}")

print("\nORDERING STATISTICS:")
print(f"  Number of orders: {simulation_result['order_count']}")
print(f"  Total units ordered: {simulation_result['total_orders']:,.2f}")
print(f"  Average order size: {simulation_result['avg_order_size']:,.2f} units")
print(
    f"  Order frequency: "
    f"{simulation_result['order_count'] / simulated_weeks:.2f} orders/week"
)

print("\nSERVICE AND SHORTAGE MEASURES:")
print(
    f"  No-shortage time-step fraction: "
    f"{simulation_result['service_level']:.2%}"
)
print(f"  Unit fill rate: {simulation_result['fill_rate']:.2%}")
print(f"  Total unmet demand: {shortage_ts.sum():,.2f} units")
print(
    f"  Time steps with unmet demand: "
    f"{simulation_result['stockout_periods']} "
    f"of {simulation_result['total_steps']:,}"
)

order_times = [index for index, quantity in enumerate(order_ts) if quantity > 0]
if order_times:
    inventory_at_order_times = [inventory_ts[index] for index in order_times]
    print("\nON-HAND INVENTORY AT ORDER TIMES:")
    print(
        f"  Mean on-hand inventory: "
        f"{np.mean(inventory_at_order_times):,.2f} units"
    )
    print(f"  First order time-step indices: {order_times[:5]}")

In [ ]:
print("=" * 72)
print("PERIODIC-REVIEW BASE-STOCK POLICY: TIER 2 SUMMARY")
print("=" * 72)

print("\nMETHOD")
print("- Evaluate candidate review periods on a finite grid.")
print("- Set base stock from the normal demand quantile for the target service level.")
print("- Compare expected weekly ordering, holding, and shortage costs.")
print("- Use a seeded finite-horizon simulation as an operational diagnostic.")

print("\nLEAST-COST GRID RESULT")
print(f"- Review period: {optimal_R:.2f} weeks")
print(f"- Associated base-stock level: {optimal_S:,.0f} units")
print(f"- Expected cost: ${minimum_cost:,.2f} per week")

print("\nCOMPARISON WITH A ONE-WEEK REVIEW PERIOD")
print(f"- One-week review expected cost: ${tier1_cost:,.2f} per week")
print(f"- Cost difference: ${cost_savings:,.2f} per week")
print(
    f"- Difference as a fraction of the one-week review cost: "
    f"{cost_savings_pct:.1f}%"
)

print("\nSIMULATION DIAGNOSTIC")
print(
    f"- No-shortage time-step fraction: "
    f"{simulation_result['service_level']:.2%}"
)
print(f"- Unit fill rate: {simulation_result['fill_rate']:.2%}")
print(f"- Mean on-hand inventory: {simulation_result['avg_inventory']:,.0f} units")
print(
    f"- Time steps with unmet demand: "
    f"{simulation_result['stockout_periods']} "
    f"of {simulation_result['total_steps']:,}"
)
print(f"- Simulated cost rate: ${sim_total_cost:,.2f} per week")

print("\nINTERPRETATION")
print(
    "The selected review period minimizes the stated expected-cost model "
    "over the tested grid. The base-stock level is determined by the "
    "service-level quantile rule. Simulation measures are finite-horizon "
    "estimates under the stated discrete-time assumptions; they need not "
    "match the steady-state expected-cost approximation or its target "
    "quantile service level."
)